<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/ner-%D0%B8-ie-%D1%81-llama-2-%D0%B8-mistral-35f64/entity_event_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!rm -rf ~/.cache/huggingface/datasets/*cuad*
!pip install -q datasets transformers accelerate bitsandbytes sentencepiece protobuf einops huggingface_hub
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q llama-cpp-python
!pip install -q pdfplumber

import json
import time
import re
from llama_cpp import Llama
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 11.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 67.4 MB/s eta 0:00:00


In [2]:
print("=" * 80)
print("ЭТАП 1: ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ")
print("=" * 80)

from datasets import load_dataset
from datasets.utils.logging import disable_progress_bar

# Отключаем прогресс-бары
disable_progress_bar()
# Загрузка датасета CUAD (используем небольшую подвыборку для CPU)
print("\nЗагрузка датасета CUAD...")
try:
    dataset = load_dataset("theatticusproject/cuad", split="train", verification_mode="no_checks")
    print(f"Всего доступно примеров: {len(dataset)}")
except Exception as e:
    print(f"Ошибка загрузки: {e}")
    # Создаем тестовые данные если загрузка не удалась
    dataset = None

# Создание подвыборки для CPU обработки (500-1000 примеров)
SAMPLE_SIZE = 500

# Берем первые SAMPLE_SIZE примеров
subset = dataset.select(range(min(SAMPLE_SIZE, len(dataset))))

print(f"Размер подвыборки: {len(subset)} примеров")

# Промпт для извлечения сущностей
EXTRACTION_PROMPT = """You are an expert legal document analyzer. Extract the following entities from the contract text:

Entities to extract:
- PERSON: Names of individuals
- ORG: Names of organizations, companies, institutions
- MONEY: Monetary amounts with currency
- DATE: Dates, deadlines, time periods
- CONTRACT_TYPE: Type of contract/agreement
- OBLIGATION: Key obligations and responsibilities
- JURISDICTION: Governing law, jurisdiction, state/country

Text: {text}

Provide the output in JSON format:
{{
  \"PERSON\": [\"name1\", \"name2\"],
  \"ORG\": [\"org1\", \"org2\"],
  \"MONEY\": [\"$1000\", \"€500\"],
  \"DATE\": [\"January 1, 2024\", \"Q1 2024\"],
  \"CONTRACT_TYPE\": [\"Service Agreement\"],
  \"OBLIGATION\": [\"obligation1\"],
  \"JURISDICTION\": [\"California\"]
}}

If no entity of a type is found, use an empty list."""

ЭТАП 1: ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

Загрузка датасета CUAD...


README.md:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

(…)62297_EX-10.33_Affiliate%20Agreement.pdf: 0.00B [00:00, ?B/s]

(…)74935_EX-10.16_Affiliate%20Agreement.pdf: 0.00B [00:00, ?B/s]

(…)anding_Atticus_Dataset_%28CUAD%29_v1.pdf: 0.00B [00:00, ?B/s]

...

(…)bution%2C%20and%20Supply%20Agreement.PDF: 0.00B [00:00, ?B/s]

(…)_22_2020-EX-10.19-SUPPLY%20AGREEMENT.PDF: 0.00B [00:00, ?B/s]

(…)98-EX-10.3-TRANSPORTATION%20CONTRACT.PDF: 0.00B [00:00, ?B/s]

(…)RANSPORTATION%20SERVICES%20AGREEMENT.PDF: 0.00B [00:00, ?B/s]

Всего доступно примеров: 511
Размер подвыборки: 500 примеров


In [3]:
print("\n" + "=" * 80)
print("ЭТАП 2: ЛОКАЛЬНОЕ РАЗВЕРТЫВАНИЕ МОДЕЛЕЙ")
print("=" * 80)

import torch

# Конфигурация для CPU
device = "cpu"
print(f"\nИспользуемое устройство: {device}")
print(f"Версия PyTorch: {torch.__version__}")

# Модели для тестирования
MODELS_CONFIG = {
    "TinyLlama-1.1B-Chat": {
        "model_id": "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        "filename": "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        "description": "Легкая модель (1.1B параметров)",
        "use_quantization": True
    },
    "Saiga-7B": {
        "model_id": "TheBloke/saiga_mistral_7b-GGUF",
        "filename": "saiga_mistral_7b.Q4_K_M.gguf",
        "description": "Saiga (7B параметров)",
        "use_quantization": True
    },
    "Mistral-7B-Quantized": {
        "model_id": "TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
        "filename": "mistral-7b-instruct-v0.2.Q4_K_M.gguf",
        "description": "Квантованная версия Mistral (7B параметров)",
        "use_quantization": True,
        "note": "Требуется llama-cpp-python для GGUF"
    }
}

print("\nКонфигурация моделей:")
for name, config in MODELS_CONFIG.items():
    print(f"  - {name}: {config['description']}")

# Функция для загрузки GGUF модели с оптимизацией для CPU
def load_model_cpu(model_id: str, filename: str, use_quantization: bool = False):
    """Загрузка GGUF модели с оптимизацией для CPU через llama-cpp-python"""
    print(f"\nЗагрузка модели: {model_id}")

    try:
        # Для GGUF моделей используем llama-cpp-python
        if use_quantization:
            print("  Используется встроенное квантование GGUF")
        else:
            print("  Загрузка с квантованием по умолчанию")

        # Загрузка модели через llama-cpp-python
        llm = Llama.from_pretrained(
            repo_id=model_id,
            filename=filename,
            n_ctx=8192,              # Размер контекстного окна
            n_threads=8,             # Количество потоков CPU
            n_gpu_layers=0,          # 0 слоев на GPU (все на CPU)
            verbose=False,
        )

        print(f"  Модель успешно загружена")
        return llm

    except Exception as e:
        print(f"  Ошибка загрузки модели: {e}")
        print("  Убедитесь, что установлена библиотека: pip install llama-cpp-python")
        return None

# Функция для генерации текста
def generate_text(llm, prompt: str, max_tokens: int = 512):
    """Генерация текста с использованием загруженной модели"""
    if llm is None:
        print("Модель не загружена")
        return None

    try:
        response = llm(
            prompt,
            max_tokens=max_tokens,
            temperature=0.1,
            top_p=0.95,
            echo=False,
            stop=["</s>", "User:", "\n\n"]
        )
        return response['choices'][0]['text']
    except Exception as e:
        print(f"Ошибка генерации: {e}")
        return None

# Загружаем только одну модель для демонстрации на CPU
print("\nЗагрузка модели для CPU...")
selected_model = "TinyLlama-1.1B-Chat"
model_config = MODELS_CONFIG[selected_model]

llm = load_model_cpu(
    model_config["model_id"],
    model_config["filename"],
    use_quantization=model_config["use_quantization"]
)

# Пример использования (раскомментируйте при необходимости)
if llm:
    print("\nМодель готова к работе!")
    # Пример генерации:
    # prompt = "Привет! Как дела?"
    # response = generate_text(llm, prompt)
    # print(f"Ответ: {response}")
else:
    print("\nНе удалось загрузить модель. Проверьте установку llama-cpp-python")

print("\n" + "=" * 80)
print("ЗАВЕРШЕНИЕ ЗАГРУЗКИ МОДЕЛИ")
print("=" * 80)


ЭТАП 2: ЛОКАЛЬНОЕ РАЗВЕРТЫВАНИЕ МОДЕЛЕЙ

Используемое устройство: cpu
Версия PyTorch: 2.10.0+cpu

Конфигурация моделей:
  - TinyLlama-1.1B-Chat: Легкая модель (1.1B параметров)
  - Saiga-7B: Saiga (7B параметров)
  - Mistral-7B-Quantized: Квантованная версия Mistral (7B параметров)

Загрузка модели для CPU...

Загрузка модели: TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF
  Используется встроенное квантование GGUF


./tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf:   0%|          | 0.00/669M [00:00<?, ?B/s]

llama_context: n_ctx_seq (8192) > n_ctx_train (2048) -- possible training context overflow


  Модель успешно загружена

Модель готова к работе!

ЗАВЕРШЕНИЕ ЗАГРУЗКИ МОДЕЛИ


In [4]:
print("\n" + "=" * 80)
print("ЭТАП 3: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ")
print("=" * 80)

# Batch processing для эффективной обработки с llama.cpp
class BatchProcessor:
    """Обработчик для пакетной обработки текстов с использованием llama.cpp"""

    def __init__(self, llm_model, batch_size: int = 4):
        self.llm = llm_model
        self.batch_size = batch_size
        self.cache = {}
        self.stats = {
            "total_processed": 0,
            "cache_hits": 0,
            "total_time": 0
        }

    def _create_cache_key(self, text: str) -> str:
        """Создание ключа для кэширования"""
        return hash(text) % 1000000

    def process_batch(self, texts: List[str]) -> List[Dict]:
        """Обработка пакета текстов"""
        start_time = time.time()

        # Проверка кэша
        cached_results = {}
        uncached_texts = []
        uncached_indices = []

        for i, text in enumerate(texts):
            cache_key = self._create_cache_key(text)
            if cache_key in self.cache:
                cached_results[i] = self.cache[cache_key]
                self.stats["cache_hits"] += 1
            else:
                uncached_texts.append(text)
                uncached_indices.append(i)

        # Обработка некэшированных текстов
        if uncached_texts:
            results = []
            for text in uncached_texts:
                prompt = EXTRACTION_PROMPT.format(text=text)
                try:
                    response = self.llm(
                        prompt,
                        max_tokens=512,
                        temperature=0.1,
                        top_p=0.95,
                        echo=False,
                        stop=["</s>", "User:", "\n\n"]
                    )
                    result = response['choices'][0]['text']
                    results.append(result)
                except Exception as e:
                    print(f"Ошибка генерации: {e}")
                    result = json.dumps({k: [] for k in ["PERSON", "ORG", "MONEY", "DATE", "CONTRACT_TYPE", "OBLIGATION", "JURISDICTION"]})
                    results.append(result)

                # Сохранение результатов в кэш
                for idx, result in zip(uncached_indices, results):
                    cached_results[idx] = result
                    cache_key = self._create_cache_key(uncached_texts[uncached_indices.index(idx)])
                    self.cache[cache_key] = result

        # Сбор результатов в правильном порядке
        results_list = [cached_results[i] for i in range(len(texts))]

        # Обновление статистики
        elapsed = time.time() - start_time
        self.stats["total_processed"] += len(texts)
        self.stats["total_time"] += elapsed

        return results_list

    def get_throughput(self) -> float:
        """Вычисление throughput (примеров в секунду)"""
        if self.stats["total_time"] == 0:
            return 0
        return self.stats["total_processed"] / self.stats["total_time"]

    def get_stats(self) -> Dict:
        """Получение статистики обработки"""
        stats = self.stats.copy()
        stats["throughput"] = self.get_throughput()
        stats["avg_time_per_sample"] = stats["total_time"] / max(stats["total_processed"], 1)
        stats["cache_hit_rate"] = stats["cache_hits"] / max(stats["total_processed"], 1)
        return stats

# Создание процессора с использованием загруженной llama.cpp модели
BATCH_SIZE = 4  # Оптимально для CPU
if llm:
    processor = BatchProcessor(llm, batch_size=BATCH_SIZE)
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Кэширование включено")
else:
    print("Ошибка: модель llm не загружена!")
    processor = None


ЭТАП 3: ОПТИМИЗАЦИЯ ПРОИЗВОДИТЕЛЬНОСТИ
Batch size: 4
Кэширование включено


In [5]:
# Ячейка 0: Принудительная установка и перезагрузка окружения
import sys

# Установка pdfplumber
!{sys.executable} -m pip install -q pdfplumber

# Перезагрузка модулей datasets для подхватывания новой библиотеки
import importlib
import datasets
importlib.reload(datasets)

# Проверка доступности
try:
    import pdfplumber
    print("✅ pdfplumber успешно установлен и загружен в этом ядре.")
except ImportError:
    print("❌ Ошибка загрузки. Попробуйте перезапустить ядро (Kernel -> Restart) и запустить эту ячейку снова.")

print("Готово к этапу 4.")

✅ pdfplumber успешно установлен и загружен в этом ядре.
Готово к этапу 4.


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 4: ИЗВЛЕЧЕНИЕ СУЩНОСТЕЙ")
print("=" * 80)

import io
import pdfplumber

# Функция парсинга JSON ответа
def parse_entities(response: str) -> Dict[str, List[str]]:
    """Парсинг JSON ответа модели"""
    try:
        json_match = re.search(r'\{.*\}', response, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            entities = json.loads(json_str)
            return entities
    except Exception as e:
        pass

    return {entity_type: [] for entity_type in ["PERSON", "ORG", "MONEY", "DATE", "CONTRACT_TYPE", "OBLIGATION", "JURISDICTION"]}

# Функция безопасного извлечения текста из элемента датасета CUAD
def safe_extract_text(item):
    """Безопасное извлечение текста из элементов датасета CUAD с PDF"""
    try:
        if isinstance(item, dict):
            # Проверяем наличие готового текстового поля
            for key in ['text', 'content']:
                if key in item and isinstance(item[key], str) and len(item[key].strip()) > 100:
                    return item[key]

            # Если есть PDF объект от datasets
            pdf_obj = item.get('pdf', None) or item.get('file', None)

            if pdf_obj is not None:
                # Объект PDF из datasets имеет атрибуты: path, bytes, или это уже декодированный объект
                pdf_bytes = None

                # Пробуем получить байты разными способами
                if hasattr(pdf_obj, 'bytes') and pdf_obj.bytes is not None:
                    pdf_bytes = pdf_obj.bytes
                elif hasattr(pdf_obj, 'read'):
                    pdf_bytes = pdf_obj.read()
                elif isinstance(pdf_obj, bytes):
                    pdf_bytes = pdf_obj
                elif hasattr(pdf_obj, 'path') and pdf_obj.path:
                    # Читаем файл по пути
                    with open(pdf_obj.path, 'rb') as f:
                        pdf_bytes = f.read()

                if pdf_bytes:
                    # Создаем BytesIO объект для pdfplumber
                    pdf_file = io.BytesIO(pdf_bytes)
                    with pdfplumber.open(pdf_file) as pdf:
                        text = ""
                        for page in pdf.pages[:5]:  # Ограничиваем первыми 5 страницами для скорости
                            page_text = page.extract_text()
                            if page_text:
                                text += page_text + "\n"
                        return text

                # Если объект уже содержит текст (некоторые версии datasets)
                if hasattr(pdf_obj, 'decode') and callable(pdf_obj.decode):
                    try:
                        return pdf_obj.decode('utf-8')
                    except:
                        pass

            # Если есть поле 'document_text' или подобное
            for key in item.keys():
                if 'text' in key.lower() and isinstance(item[key], str):
                    return item[key]

        # Попытка через .get для объектов Dataset
        if hasattr(item, 'get'):
            text = item.get('text', None)
            if text and len(str(text).strip()) > 100:
                return str(text)

        return None

    except Exception as e:
        print(f"  Ошибка извлечения текста: {e}")
        return None

# Извлечение текстов из датасета
texts_to_process = []
print("\nИзвлечение текстов из датасета...")

print("\nОбработка элементов...")
success_count = 0
error_count = 0

for i in range(min(50, len(subset))):
    item = subset[i]
    text = safe_extract_text(item)

    if text and len(text.strip()) > 100:
        texts_to_process.append(text)
        success_count += 1
    else:
        error_count += 1

print(f"\n{'='*60}")
print(f"Успешно извлечено: {success_count} документов")
print(f"Пропущено: {error_count} документов")
print(f"Готово к обработке: {len(texts_to_process)} текстов")
print(f"{'='*60}")


# Пакетная обработка с использованием llama.cpp
print("\nНачало обработки текстов моделью...")
start_time = time.time()

all_results = []
if processor:
    for i in range(0, len(texts_to_process), BATCH_SIZE):
        batch = texts_to_process[i:i+BATCH_SIZE]
        responses = processor.process_batch(batch)

        for text, response in zip(batch, responses):
            entities = parse_entities(response)
            all_results.append({
                "text": text[:300] + "..." if len(text) > 300 else text,
                "full_text": text,
                "entities": entities,
                "raw_response": response
            })

        progress = min(i + BATCH_SIZE, len(texts_to_process))
        print(f"  Обработано {progress}/{len(texts_to_process)} примеров")

    total_time = time.time() - start_time
    print(f"\n✅ Обработка завершена за {total_time:.2f} секунд")
    print(f"Всего обработано документов: {len(all_results)}")
else:
    print("❌ Ошибка: процессор не инициализирован!")


ЭТАП 4: ИЗВЛЕЧЕНИЕ СУЩНОСТЕЙ

Извлечение текстов из датасета...

Обработка элементов...

Успешно извлечено: 50 документов
Пропущено: 0 документов
Готово к обработке: 50 текстов

Начало обработки текстов моделью...
  Обработано 4/50 примеров
  Обработано 8/50 примеров
  Обработано 12/50 примеров


In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 6: BENCHMARK МОДЕЛЕЙ")
print("=" * 80)

import os
import psutil
import time
import json
import re
from collections import defaultdict

# --- Конфигурация бенчмарка ---
BENCHMARK_SAMPLES = 10  # Количество текстов для теста
WARMUP_SAMPLES = 2      # Прогрев (не учитывается в метриках)

def run_benchmark(processor, texts):
    """Запуск бенчмарка производительности"""
    print(f"\n🚀 Запуск бенчмарка на {BENCHMARK_SAMPLES} примерах...")

    # Прогрев
    if len(texts) > WARMUP_SAMPLES:
        print("  Выполнение прогрева модели...")
        processor.process_batch(texts[:WARMUP_SAMPLES])

    # Основной прогон
    test_texts = texts[WARMUP_SAMPLES:BENCHMARK_SAMPLES]
    if not test_texts:
        test_texts = texts[:BENCHMARK_SAMPLES]

    start_time = time.time()

    results = []
    for i in range(0, len(test_texts), processor.batch_size):
        batch = test_texts[i:i+processor.batch_size]
        _ = processor.process_batch(batch)

    end_time = time.time()
    total_time = end_time - start_time

    # Сбор метрик
    stats = processor.get_stats()
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()

    return {
        "total_time": total_time,
        "samples_processed": len(test_texts),
        "throughput": len(test_texts) / total_time if total_time > 0 else 0,
        "avg_latency_ms": (total_time / len(test_texts) * 1000) if len(test_texts) > 0 else 0,
        "memory_mb": memory_info.rss / 1024 / 1024,
        "cpu_count": psutil.cpu_count(logical=True),
        "threads_used": getattr(processor.llm, 'n_threads', 'N/A')
    }

# Подготовка данных для бенчмарка
benchmark_texts = []
for i in range(min(BENCHMARK_SAMPLES + WARMUP_SAMPLES, len(subset))):
    item = subset[i]
    # Безопасное извлечение текста (упрощенная версия для бенчмарка)
    text = ""
    if isinstance(item, dict):
        text = item.get('text', item.get('content', ''))
    if isinstance(text, bytes):
        try: text = text.decode('utf-8')
        except: text = ""

    if text and len(text) > 50:
        # Тримминг для стабильности теста, чтобы не вылетать по контексту
        benchmark_texts.append(text[:2000])

if len(benchmark_texts) < 2:
    print("⚠️ Недостаточно данных для бенчмарка. Пропуск.")
    benchmark_results = None
else:
    # Запуск
    metrics = run_benchmark(processor, benchmark_texts)

    print("\n📊 РЕЗУЛЬТАТЫ BENCHMARK:")
    print("-" * 50)
    print(f"Модель: Saiga-7B (GGUF)")
    print(f"Потоков CPU: {metrics['threads_used']}")
    print(f"Всего образцов: {metrics['samples_processed']}")
    print(f"Общее время: {metrics['total_time']:.2f} сек")
    print(f"Throughput: {metrics['throughput']:.2f} текстов/сек")
    print(f"Средняя задержка: {metrics['avg_latency_ms']:.2f} мс/текст")
    print(f"Потребление RAM: {metrics['memory_mb']:.1f} MB")
    print(f"Логических ядер CPU: {metrics['cpu_count']}")

    benchmark_results = metrics

In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 6: BENCHMARK МОДЕЛЕЙ")
print("=" * 80)

import os
import psutil
import time
import json
import re
from collections import defaultdict

# --- Конфигурация бенчмарка ---
BENCHMARK_SAMPLES = 10  # Количество текстов для теста
WARMUP_SAMPLES = 2      # Прогрев (не учитывается в метриках)

def run_benchmark(processor, texts):
    """Запуск бенчмарка производительности"""
    print(f"\n🚀 Запуск бенчмарка на {BENCHMARK_SAMPLES} примерах...")

    # Прогрев
    if len(texts) > WARMUP_SAMPLES:
        print("  Выполнение прогрева модели...")
        processor.process_batch(texts[:WARMUP_SAMPLES])

    # Основной прогон
    test_texts = texts[WARMUP_SAMPLES:BENCHMARK_SAMPLES]
    if not test_texts:
        test_texts = texts[:BENCHMARK_SAMPLES]

    start_time = time.time()

    results = []
    for i in range(0, len(test_texts), processor.batch_size):
        batch = test_texts[i:i+processor.batch_size]
        _ = processor.process_batch(batch)

    end_time = time.time()
    total_time = end_time - start_time

    # Сбор метрик
    stats = processor.get_stats()
    process = psutil.Process(os.getpid())
    memory_info = process.memory_info()

    return {
        "total_time": total_time,
        "samples_processed": len(test_texts),
        "throughput": len(test_texts) / total_time if total_time > 0 else 0,
        "avg_latency_ms": (total_time / len(test_texts) * 1000) if len(test_texts) > 0 else 0,
        "memory_mb": memory_info.rss / 1024 / 1024,
        "cpu_count": psutil.cpu_count(logical=True),
        "threads_used": getattr(processor.llm, 'n_threads', 'N/A')
    }

# Подготовка данных для бенчмарка
benchmark_texts = []
for i in range(min(BENCHMARK_SAMPLES + WARMUP_SAMPLES, len(subset))):
    item = subset[i]
    # Безопасное извлечение текста (упрощенная версия для бенчмарка)
    text = ""
    if isinstance(item, dict):
        text = item.get('text', item.get('content', ''))
    if isinstance(text, bytes):
        try: text = text.decode('utf-8')
        except: text = ""

    if text and len(text) > 50:
        # Тримминг для стабильности теста, чтобы не вылетать по контексту
        benchmark_texts.append(text[:2000])

if len(benchmark_texts) < 2:
    print("⚠️ Недостаточно данных для бенчмарка. Пропуск.")
    benchmark_results = None
else:
    # Запуск
    metrics = run_benchmark(processor, benchmark_texts)

    print("\n📊 РЕЗУЛЬТАТЫ BENCHMARK:")
    print("-" * 50)
    print(f"Модель: Saiga-7B (GGUF)")
    print(f"Потоков CPU: {metrics['threads_used']}")
    print(f"Всего образцов: {metrics['samples_processed']}")
    print(f"Общее время: {metrics['total_time']:.2f} сек")
    print(f"Throughput: {metrics['throughput']:.2f} текстов/сек")
    print(f"Средняя задержка: {metrics['avg_latency_ms']:.2f} мс/текст")
    print(f"Потребление RAM: {metrics['memory_mb']:.1f} MB")
    print(f"Логических ядер CPU: {metrics['cpu_count']}")

    benchmark_results = metrics

In [ ]:
print("\n" + "=" * 80)
print("ЭТАП 7: ПАРАЛЛЕЛЬНАЯ ОБРАБОТКА")
print("=" * 80)

from multiprocessing import Process, Queue, cpu_count
import pickle

# Функция-обертка для запуска в отдельном процессе
# Примечание: В реальном продакшене модель нужно загружать внутри процесса,
# так как объекты llama_cpp не всегда корректно сериализуются (pickle).
# Для демонстрации мы эмулируем распределение нагрузки.

def worker_process(task_queue, result_queue, model_config):
    """
    Worker процесс.
    ВНИМАНИЕ: Для реальной работы здесь должна быть инициализация Llama(...)
    внутри процесса, так как модель нельзя передать через Queue напрямую.
    """
    # Эмуляция загрузки модели в процессе (псевдокод для структуры)
    # from llama_cpp import Llama
    # llm_local = Llama.from_pretrained(...)

    while True:
        task = task_queue.get()
        if task is None:  # Сигнал завершения
            break

        idx, text = task
        # Здесь была бы логика генерации:
        # output = llm_local(prompt, ...)

        # Эмуляция результата для демонстрации структуры
        result = {
            "id": idx,
            "status": "processed",
            "entities": {"PERSON": ["Demo"], "ORG": ["DemoCorp"]}
        }
        result_queue.put(result)

# Настройка параллелизма
NUM_WORKERS = max(1, cpu_count() - 1)  # Оставляем 1 ядро системе
print(f"\n⚙️ Конфигурация параллельности:")
print(f"  Доступно ядер CPU: {cpu_count()}")
print(f"  Запускаем воркеров: {NUM_WORKERS}")

# Подготовка задач
tasks = []
for i, text in enumerate(benchmark_texts[:10]): # Берем немного для демо
    tasks.append((i, text))

print(f"  Подготовлено задач: {len(tasks)}")

# Реализация через Process Pool (упрощенная симуляция для безопасности окружения)
# В реальном коде здесь был бы multiprocessing.Pool или Process

print("\n🔄 Запуск параллельной обработки...")
start_parallel = time.time()

# Поскольку полная инициализация модели в каждом процессе требует
# повторной загрузки весов (что медленно для демо),
# мы покажем расчетный выигрыш и структуру кода.

# Расчетный теоретический выигрыш
single_thread_time = len(tasks) / benchmark_results['throughput'] if benchmark_results else 0
# Коэффициент эффективности параллелизма на CPU обычно 0.6-0.8 из-за накладных расходов
estimated_speedup = min(NUM_WORKERS, 4) * 0.7
estimated_parallel_time = single_thread_time / estimated_speedup if estimated_speedup > 0 else 0

print(f"\n📈 ПРОГНОЗ ПРОИЗВОДИТЕЛЬНОСТИ:")
print("-" * 50)
print(f"Последовательная обработка ({len(tasks)} текстов): ~{single_thread_time:.2f} сек")
print(f"Параллельная обработка ({NUM_WORKERS} воркеров): ~{estimated_parallel_time:.2f} сек (оценка)")
print(f"Ожидаемое ускорение: в {estimated_speedup:.1f} раз")

print("\n💡 РЕКОМЕНДАЦИЯ ДЛЯ LLAMA.CPP:")
print("Для максимальной производительности на CPU используйте:")
print("1. Параметр n_threads = количеству физических ядер.")
print("2. Библиотеку llama-cpp-python с поддержкой OpenBLAS/MKL.")
print("3. Избегайте чрезмерного параллелизма процессов, если память ограничена (каждая копия модели ест RAM).")

end_parallel = time.time()
print(f"\nДемо завершено за {end_parallel - start_parallel:.2f} сек")

In [ ]:
print("\n" + "=" * 80)
print("ИТОГ")
print("=" * 80)

print("\n📊 ИТОГОВЫЕ МЕТРИКИ:")
print(f"  • Обработано документов: {len(all_results)}")
print(f"  • Извлечено сущностей: {total_entities}")
print(f"  • Throughput: {stats['throughput']:.2f} примеров/сек")
print(f"  • Средняя латентность: {stats['avg_time_per_sample']*1000:.2f} мс")
print(f"  • Потребление памяти: {memory_mb:.1f} MB")